# 🏠 Predicting and Explaining Austin House Prices

Welcome! This notebook upgrades a baseline Linear Regression model into an interpretable and presentation-ready workflow. We will:

- Build a reproducible ML pipeline (preprocess → train → predict → evaluate)
- Explain predictions using model coefficients and SHAP
- Visualize key drivers: feature impacts, actual vs predicted, residuals, and SHAP plots

We'll use `austin_properties_cleaned.csv` as our primary dataset and keep the model simple and transparent with Linear Regression.


## 1️⃣ Introduction
This notebook demonstrates an explainable Linear Regression pipeline for Austin property prices. We'll quantify global feature impacts via coefficients and SHAP, and provide a per-property explanation.


## 2️⃣ Data Loading and Preparation
We load the cleaned dataset, select numeric features, split into train/test, and standardize features for stable coefficient interpretation. Target is `latestPrice`.


In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Plotting style
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

# Load data
csv_path = 'austin_properties_cleaned.csv'  # same folder
raw = pd.read_csv(csv_path)

# Basic cleaning: drop obvious non-numeric or leakage columns for baseline
candidate_target = 'latestPrice'
non_feature_cols = {
    'zpid','city','streetAddress','description','homeImage','latest_saledate',
    'homeType'  # keep it simple for Linear Regression
}

# Keep numeric columns only; ensure target present
numeric_cols = raw.select_dtypes(include=[np.number]).columns.tolist()
assert candidate_target in numeric_cols, 'latestPrice must be numeric and present in dataset'

# Remove target from features and any ID-like fields if present
feature_cols = [c for c in numeric_cols if c != candidate_target]

# Drop rows with missing target
df = raw.dropna(subset=[candidate_target]).copy()
# Simple impute for features: median
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

X = df[feature_cols]
y = df[candidate_target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

feature_names = feature_cols
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


## 3️⃣ Model Training (Linear Regression)
We train a simple Linear Regression on standardized features to obtain interpretable coefficients.


In [ ]:
# Train model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = lr.predict(X_train_scaled)
y_test_pred = lr.predict(X_test_scaled)

# Evaluation
r2 = r2_score(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)

print(f"R^2 (test): {r2:.3f}")
print(f"RMSE (test): ${rmse:,.0f}")


## 4️⃣ Evaluation Metrics (R², MSE)
We report R² and RMSE to assess performance. Then we visualize actual vs predicted and residuals to spot biases or heteroscedasticity.


In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots()
sns.scatterplot(x=y_test, y=y_test_pred, ax=ax)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Ideal')
ax.set_xlabel('Actual Price')
ax.set_ylabel('Predicted Price')
ax.set_title('Actual vs Predicted Prices')
ax.legend()
plt.show()

# Residuals
residuals = y_test - y_test_pred
fig, ax = plt.subplots()
sns.histplot(residuals, kde=True, ax=ax)
ax.set_title('Residuals Distribution')
ax.set_xlabel('Residual (Actual - Predicted)')
plt.show()

print('Interpretation: Points close to the dashed line indicate accurate predictions. A roughly symmetric residual distribution suggests no major bias.')


## 5️⃣ Feature Importance Analysis
We inspect standardized coefficients. Positive coefficients increase predicted price; negative decrease it. We'll show the top features by absolute impact.


In [ ]:
# Coefficients (standardized)
coef_series = pd.Series(lr.coef_, index=feature_names)
coef_sorted = coef_series.reindex(coef_series.abs().sort_values(ascending=False).index)

# Top 20 features
top_k = 20
coef_top = coef_sorted.head(top_k)

fig, ax = plt.subplots(figsize=(10, max(6, top_k * 0.35)))
sns.barplot(x=coef_top.values, y=coef_top.index, ax=ax, palette=['#2ca02c' if v>0 else '#d62728' for v in coef_top.values])
ax.set_title('Feature Impact (Linear Regression Coefficients)')
ax.set_xlabel('Coefficient (Standardized)')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

print('Interpretation: Green bars push price higher; red bars lower it. Longer bars imply stronger influence.')


## 6️⃣ Explainability using SHAP
We compute SHAP values on the standardized features for global and local explanations. We'll include a summary plot and a single-instance waterfall plot.


In [ ]:
# SHAP explanations
# For linear models, KernelExplainer works but LinearExplainer is efficient & exact for linear links.
explainer = shap.LinearExplainer(lr, X_train_scaled, feature_names=feature_names)
shap_values_test = explainer.shap_values(X_test_scaled)

# Global summary plot
shap.summary_plot(shap_values_test, features=X_test_scaled, feature_names=feature_names, show=False)
plt.title('SHAP Summary: Global Feature Influence')
plt.tight_layout()
plt.show()

# Single prediction explanation (first test sample)
sample_idx = 0
shap.plots._waterfall.waterfall_legacy(
    shap.Explanation(values=shap_values_test[sample_idx],
                     base_values=explainer.expected_value,
                     data=X_test_scaled[sample_idx],
                     feature_names=feature_names)
)
plt.title('Why this property price? Top drivers (waterfall)')
plt.tight_layout()
plt.show()

print('Interpretation: SHAP summarizes how each feature pushes the prediction above or below the baseline expected value.')


## 7️⃣ Visualization Dashboard (Optional Interactive)
We include an optional Plotly scatter for Actual vs Predicted with hover details for presentation.


In [ ]:
# Plotly Actual vs Predicted
try:
    df_plot = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_test_pred})
    fig = px.scatter(df_plot, x='Actual', y='Predicted', title='Interactive: Actual vs Predicted', trendline='ols')
    fig.show()
    print('Interpretation: Interactive plot helps explore over/under-prediction regions.')
except Exception as e:
    print(f'Plotly interactive plot skipped: {e}')


## 8️⃣ Interpretation of Results
- R² and RMSE quantify overall performance.
- Coefficient bar chart shows the strongest drivers (positive/negative) at the global level.
- SHAP summary validates global drivers and the waterfall plot explains a single property in detail.


## 9️⃣ Conclusion and Future Work
- The Linear Regression model is transparent and provides clear global and local explanations.
- Future enhancements:
  - Feature engineering (e.g., interactions, non-linear transforms)
  - Compare with regularized models (Ridge/Lasso) and tree-based models
  - Use cross-validation and calibration
  - Add LIME explanations alongside SHAP for robustness
